# Foundations of Data Science – Project Notebook

**Course:** Foundations of Data Science  
**Authors:** Maja Skwaroń (66083), Jagoda Seidel (66090), Maria João Vicente (44489)  
**Date:** 10/12/2025  

This notebook documents the full workflow of the project, from data loading and cleaning to model training and interpretation.

**Core libraries and configurations**

In [100]:
# Uncomment this command to install required python libraries
# %pip install -r requirements.txt

In [ ]:
import copy
import os
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import tqdm
from lime.lime_tabular import LimeTabularExplainer
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeRegressor, plot_tree
from ydata_profiling import ProfileReport

from src.nn_model import train_neural_network, visualize_3_param_grid_search_results

%matplotlib inline

ImportError: cannot import name 'visualize_3_param_grid_search_results' from 'src.nn_model' (c:\Users\mjvsilva\Documents\_mystuff\FDS\project_group_2_fds_202526\src\nn_model.py)

In [ ]:
# Create directories where outputs will be saved

os.makedirs("data", exist_ok=True)
os.makedirs("data/outputs", exist_ok=True)
os.makedirs("data/reports", exist_ok=True)
os.makedirs("models", exist_ok=True)

In [ ]:
# Set random seeds for reproducibility

random_seed_value = 48

torch.manual_seed(random_seed_value)
np.random.seed(random_seed_value)

## 1. Raw Data Loading and Exploration

### 1.1 Loading Datasets

In [ ]:
wage_gap = pd.read_csv("data/inputs/wage_gap.csv")  # OECD “Gender Wage Gap”
GIS = pd.read_csv("data/inputs/Gender_Institutions_Development.csv")  # OECD “Gender, Institutions and Development Database (GID-DB) 2023”
education = pd.read_csv("data/inputs/Education.csv")  # OECD “Adults' educational attainment distribution, by age group and gender”
employment = pd.read_csv("data/inputs/Employment rate.csv")  # OECD “Employment rates of adults, by educational attainment, age group and gender”
lfp = pd.read_csv("data/inputs/Labour_force_participation.csv")  # OECD “Labour market outcomes of immigrants - Employment, unemployment, and participation rates by sex”
GDP = pd.read_csv("data/inputs/GDP and consuption per capita.csv")  # OECD “Annual GDP and consumption per capita - multilateral indices”
pt_employment = pd.read_csv("data/inputs/part-time_employment.csv")  # OCED “Incidence of full-time and part-time employment based on OECD-harmonized definition”
p_e_representation = pd.read_csv("data/inputs/public_employment_representation.csv")  # OCED “Public employment and representation - government at a glance indicators, yearly updates”

### 1.2 Exploration

In this section we explore the raw data. The code below was used to create the tables presented in Appendix B of the report.

To gain a clearer understanding of the structure of each dataset, we computed the number of observations and variables, as well as the first and last years for which data were available. Additionally, we calculated the number of distinct countries represented in each dataset.

The “Gender, Institutions and Development Database (GID-DB) 2023” and the “Public employment and representation – government at a glance” datasets contain a large number of indicators that are relevant to our analysis. Due to their size, it was not feasible to display all variables in the raw data samples presented in Appendix A. Therefore, supplementary tables (B2 and B3) were created in Appendix B to list these indicators together with their corresponding units of measure.

In [ ]:
# A function to calculates number of observations, number of variables, first and last year, number of distinct countries
def dataset_stats(df):
    return {
        "Number of observations": len(df),
        "Number of variables": df.shape[1],
        "First year available": df["TIME_PERIOD"].min() if "TIME_PERIOD" in df.columns else None,
        "Last year available": df["TIME_PERIOD"].max() if "TIME_PERIOD" in df.columns else None,
        "Number of distinct countries": df["REF_AREA"].nunique() if "REF_AREA" in df.columns else None
    }

In [ ]:
# Applying a function on our data (Table B1)
datasets = {
    "Gender Wage Gap": wage_gap,
    "Adults' educational attainment distribution, by age group and gender": education,
    "Employment rates of adults, by educational attainment, age group and gender": employment,
    "Labour market outcomes of immigrants - Employment, unemployment, and participation rates by sex": lfp,
    "Incidence of full-time and part-time employment based on OECD-harmonized definition": pt_employment,
    "Annual GDP and consumption per capita - multilateral indices": GDP,
    "Gender, Institutions and Development": GIS,
    "Public employment and representation - government at a glance indicatorsn": p_e_representation
}

summary = {name: dataset_stats(df) for name, df in datasets.items()}

print(pd.DataFrame(summary).T)

In [ ]:
# Table B2
subset = GIS[['Measure', 'Unit of measure']]
unique_pairs = subset.drop_duplicates().reset_index(drop=True)

unique_pairs

In [ ]:
# Table B3
subset = p_e_representation[['Measure', 'Unit of measure']]
unique_pairs = subset.drop_duplicates().reset_index(drop=True)

unique_pairs

## 2. Data Processing

All datasets were transformed into a consistent format, where each raw represented a country-year combination. For each dataset, only the relevant variables were retained, and columns were renamed to a common schema, for easier processing. Some indicators had to be derived from available variables, and the process of calculating the new values is provided under this section. The cleaned datasets were then sorted and saved for later integration. 


### 2.1 OECD “Gender Wage Gap”

In [ ]:
# Keep only needed columns
df_sub = wage_gap[[
    "Reference area",
    "TIME_PERIOD",
    "OBS_VALUE"
]].copy()

# Rename columns
df_sub = df_sub.rename(columns={
    "Reference area": "country",
    "TIME_PERIOD": "year",
    "OBS_VALUE": "wage_gap"
})

print(df_sub.groupby(by=["country", "year"], as_index=False)["wage_gap"].nunique()["wage_gap"].unique())

# Make sure country-year is unique
df_sub = df_sub.drop_duplicates(subset=["country", "year"])

# Sort for clarity
df_sub = df_sub.sort_values(["country", "year"]).reset_index(drop=True)

#saving transformed dataset
df_sub.to_csv("data/outputs/wage_gap_transformed.csv", index=False)

### 2.2 OECD “Gender, Institutions and Development Database (GID-DB) 2023

For this dataset besides regular transformation to country-year format, standarizing column names, and dropping irrelevant columns, separating indicators that appeared with different units or for different sexes, and reshaping the data into a wide format was needed.  Additionally, a new variables unpaid care gap (men - women, in hours per day) was computed.

In [ ]:
# Keep only needed columns
df_subset = GIS[[
    "Reference area",
    "TIME_PERIOD",
    "Measure",
    "Unit of measure",
    "Age",
    "Sex",
    "OBS_VALUE"
]].copy()

# Rename columns
df_subset = df_subset.rename(columns={
    "Reference area": "country",
    "TIME_PERIOD": "year",
    "Measure": "indicator",
    "Unit of measure": "unit",
    "Age": "age",
    "Sex": "sex",
    "OBS_VALUE": "value"
})

# Clean weird spaces in indicator names
df_subset['indicator'] = df_subset['indicator'].str.replace('\xa0', ' ', regex=False)

# Define special indicators that need sex + unit split
special_indicators = [
    'Lack of confidence in the judicial system and courts',
    'Time spent on unpaid care and domestic work'
]

def build_colname(row):
    if row['indicator'] in special_indicators:
        # Separate column for each sex + unit combination
        return f"{row['indicator']} - {row['sex']} - {row['unit']}"
    else:
        # All other indicators: just one column
        return row['indicator']

df_subset['colname'] = df_subset.apply(build_colname, axis=1)

# Pivot to wide: one row per country-year
df_wide = (
    df_subset
    .pivot_table(
        index=["country", "year"],
        columns="colname",
        values="value",
        aggfunc="first"
    )
    .reset_index()
)

# Compute unpaid care gap: man - woman (hours per day)
df_wide['unpaid_care_gap'] = (
    df_wide['Time spent on unpaid care and domestic work - Male - Hours per day'] -
    df_wide['Time spent on unpaid care and domestic work - Female - Hours per day']
)


# Keep only the relevant columns
selected_columns = [
    "country",
    "year",
    "Gender gap in managerial positions",
    "Gender gap in political representation",
    "Gender gap in top management positions",
    "Lack of confidence in the judicial system and courts - Female - Percentage of population",
    "Legal discrimination on access to justice",
    "Legal discrimination on financial assets",
    "Legal discrimination on freedom of movement",
    "Legal discrimination on inheritance",
    "Legal discrimination on violence against women",
    "Legal discrimination on workplace rights",
    "Perception that a woman earning more money than her husband causes problems",
    "Perception that being a housewife is as fulfilling as working for pay",
    "Perception that men make better political leaders than women",
    "Perception that men should have more right to a job than women",
    "Perception that when a mother works for pay, the children suffer",
    "Perceptions that men make better business executives than women",
    "unpaid_care_gap"
]

existing_cols = [c for c in selected_columns if c in df_wide.columns]
df_model = df_wide[existing_cols].copy()

# Save dataset for merging with other data
df_model.to_csv("data/outputs/GIS_gap.csv", index=False)


### 2.3 OECD “Adults' educational attainment distribution, by age group and gender” 

For this dataset data was reshaped into a country–year format, and the gender education gap was calculated as the difference between men’s and women’s tertiary attainment.

In [ ]:
# Keep only relevant columns
df_small = education[[
    "Reference area",
    "TIME_PERIOD",
    "SEX",
    "OBS_VALUE"
]].copy()

# Rename columns
df_small = df_small.rename(columns={
    "Reference area": "country",
    "TIME_PERIOD": "year",
    "SEX": "sex",
    "OBS_VALUE": "value"
})

# Pivot to wide format
df_wide = (
    df_small.pivot_table(
        index=["country", "year"],
        columns="sex",
        values="value"
    )
    .reset_index()
)

# Rename columns from Sex codes
df_wide = df_wide.rename(columns={
    "F": "women_tertiary",
    "M": "men_tertiary"
})

# Calculate gender education gap
df_wide["education_gap"] = df_wide["men_tertiary"] - df_wide["women_tertiary"]

# Select only the relevant variables
df_gap_only = df_wide[["country", "year", "education_gap"]].copy()

# Save to CSV
df_gap_only.to_csv("data/outputs/Education_gap.csv", index=False)


### 3.4 OECD “Employment rates of adults, by educational attainment, age group and gender”

For this dataset data was reshaped into a country–year format, and the gender employment gap was calculated as the difference between men’s and women’s employment rate.

In [ ]:
# Keep only the needed columns
df_subset = employment[[
    "Reference area",
    "Sex",
    "TIME_PERIOD",
    "OBS_VALUE"
]].copy()

# Rename
df_subset = df_subset.rename(columns={
    "Reference area": "country",
    "Sex": "sex",
    "TIME_PERIOD": "year",
    "OBS_VALUE": "employment_rate"
})

# Pivot: one row per country-year, separate columns for men and women
employment_wide = (
    df_subset
    .pivot_table(
        index=["country", "year"],
        columns="sex",
        values="employment_rate"
    )
    .reset_index()
    .rename_axis(None, axis=1)
)

# Rename columns
employment_wide = employment_wide.rename(columns={
    "Female": "employment_women",
    "Male": "employment_men"
})

# Calculate gender employment gap
employment_wide["employment_gap"] = (
    employment_wide["employment_men"] - employment_wide["employment_women"]
)

# Select only relevant columns
df_gap_only = employment_wide[["country", "year", "employment_gap"]].copy()

# Save to CSV
df_gap_only.to_csv("data/outputs/Employment_gap.csv", index=False)


### 3.5 OECD “Labour market outcomes of immigrants - Employment, unemployment, and participation rates by sex”

The data was reshaped, and average labour-force participation rates were calculated for men and women across native-born and foreign-born groups. Additionally, gender gaps (men - women) for natives, immigrants, and the overall population were calculated.

In [ ]:
# Keep only the relevant columns
df = lfp[["Reference area", "SEX", "TIME_PERIOD", "Place of birth", "OBS_VALUE"]]

# Rename
df = df.rename(columns={
    "Reference area": "country",
    "SEX": "sex",
    "TIME_PERIOD": "year",
    "Place of birth": "birthplace",
    "OBS_VALUE": "lfpr"
})

# Keep only Male/Female and the two birthplace groups (exclude totals)
sex_map = {"F": "Female", "Female": "Female", "M": "Male", "Male": "Male"}
birth_map = {
    "Native-born": "Native-born",
    "Foreign-born": "Foreign-born",
    "Native born": "Native-born",
    "Foreign born": "Foreign-born",
}

df["sex"] = df["sex"].map(lambda x: sex_map.get(x, x))
df["birthplace"] = df["birthplace"].map(lambda x: birth_map.get(x, x))

df2 = df[
    df["sex"].isin(["Female", "Male"]) # filter only for rows where sex is either female or male (delete total)
    & df["birthplace"].isin(["Native-born", "Foreign-born"])
][["country", "year", "sex", "birthplace", "lfpr"]] # select only these columns for the new data frame

# Create the target column names
name_map = {
    ("Female", "Native-born"): "woman_native",
    ("Male", "Native-born"): "man_native",
    ("Female", "Foreign-born"): "woman_immigrant",
    ("Male", "Foreign-born"): "man_immigrant",
}
df2["colname"] = df2.apply(lambda r: name_map[(r["sex"], r["birthplace"])], axis=1)

# Wide table: one row per country-year, desired columns as values
wide = (
    df2.pivot_table(index=["country", "year"], columns="colname", values="lfpr")
    .reset_index()
    .rename_axis(None, axis=1)
)

# Reorder columns
ordered = ["country", "year", "woman_native", "man_native", "woman_immigrant", "man_immigrant"]
wide = wide[[c for c in ordered if c in wide.columns]]

# Calculate the combined average for women and men (across native & immigrant)
wide["woman_avg"] = wide[["woman_native", "woman_immigrant"]].mean(axis=1)
wide["man_avg"]   = wide[["man_native", "man_immigrant"]].mean(axis=1)

# Calculate gender participation gaps
wide["native_gap"] = wide["man_native"] - wide["woman_native"]
wide["immigrant_gap"] = wide["man_immigrant"] - wide["woman_immigrant"]
wide["overall_gap"] = wide["man_avg"] - wide["woman_avg"]

# Save only relevant columns
gap_dataset = wide[[
    "country",
    "year",
    "native_gap",
    "immigrant_gap",
    "overall_gap"
]].copy()

# Save to CSV
gap_dataset.to_csv("data/outputs/lfp_gap.csv", index=False)


### 3.6 OECD “Annual GDP and consumption per capita - multilateral indices”

In [ ]:
# Keep only relevant columns
df_clean = GDP[[
    "Reference area", "Transaction", "Unit of measure",
    "Price base", "TIME_PERIOD", "OBS_VALUE"
]].copy()

# Rename for easier use
df_clean.columns = ["country", "transaction", "unit", "price_base", "year", "value"]

# Pivot: one row per country-year
df_wide = (
    df_clean
    .pivot_table(index=["country", "year"], columns="transaction", values="value")
    .reset_index()
)

# Rename columns
df_wide.columns.name = None
df_wide = df_wide.rename(columns={
    "Gross domestic product, per capita": "gdp_pc",
    "Actual individual consumption, per capita": "consumption_pc_ppp_oecd_pct"
})

# Keep only relevant columns
gdp_only = df_wide[["country", "year", "gdp_pc"]].copy()

# Save it
gdp_only.to_csv("data/outputs/GDP.csv", index=False)


### 3.7 OCED “Incidence of full-time and part-time employment based on OECD-harmonized definition”

For this dataset data was reshaped into a country–year format. The gender part-time employment gap was calculated as the difference between men’s and women’s part-time employment rate.

In [ ]:
# Keep only relevant columns
df_clean = pt_employment[[
    "Reference area",
    "TIME_PERIOD",
    "Sex",
    "OBS_VALUE"
]].copy()

# Rename
df_clean = df_clean.rename(columns={
    "Reference area": "country",
    "TIME_PERIOD": "year",
    "Sex": "sex",
    "OBS_VALUE": "value"
})

# Pivot to wide format: one row per country–year
df_wide = (
    df_clean
    .pivot_table(index=["country", "year"],
                 columns="sex",
                 values="value")
    .reset_index()
)

# Rename columns
df_wide = df_wide.rename(columns={
    "Female": "part_time_women",
    "Male": "part_time_men"
})

# compute gender gap
df_wide["part_time_gap"] = df_wide["part_time_men"] - df_wide["part_time_women"]

# Select only relevant columns
df_gap_only = df_wide[["country", "year", "part_time_gap"]].copy()

# Save to CSV
df_gap_only.to_csv("data/outputs/Part-time_employment_gap.csv", index=False)

### 3.8  OCED “Public employment and representation - government at a glance indicators, yearly updates”

In this dataset the data was not available for man and woman separately but only as a measure of equality (% of woman). To stay consistent with the rest of the indicators that will be included in our analysis, the gap between man and woman for indicators in this dataset was calculated as 100 - 2 * (% women), since % man = 100% - % women.

In [ ]:
# Keep only relevant columns
df_sub = p_e_representation[[
    "Reference area",
    "Measure",
    "TIME_PERIOD",
    "OBS_VALUE"
]].copy()

# Rename
df_sub = df_sub.rename(columns={
    "Reference area": "country",
    "Measure": "indicator",
    "TIME_PERIOD": "year",
    "OBS_VALUE": "value"
})

# Pivot to wide format: one row per country–year, columns = indicators
wide = (
    df_sub
    .pivot_table(index=["country", "year"], columns="indicator", values="value")
    .reset_index()
    .rename_axis(None, axis=1)
)

# Map of original indicator names -> desired gap column names
gap_map = {
    "Gender equality in parliament": "Gender gap in representation in parliament",
    "Gender equality in professional judges": "Gender gap in professional judges representation",
    "Gender equality in public sector employment": "Gender gap in public sector employment ",
    "Gender equality in senior management positions in national administrations":
        "Gender gap in senior management positions in national administrations",
}

# Calculate gaps: gap = 100 - 2 * (% women)
for src_col, gap_col in gap_map.items():
    if src_col in wide.columns:
        wide[gap_col] = 100 - 2 * wide[src_col]

# Keep only relevant columns
final_cols = ["country", "year"] + [
    gap_name for src, gap_name in gap_map.items() if gap_name in wide.columns
]
wide_final = wide[final_cols]

wide_final.to_csv("data/outputs/public_employment_representation_gap.csv", index=False)

## 3. Data Merging

### 3.1 Expanding the Dataset to a Full Country–Year Grid

Before merging different data sources, the Gender Wage Gap dataset was first expanded into a complete country–year panel to ensure that every country included in the analysis had an entry for every year in the expected range (1970–2024). If data was not available for a given year, a given observation was becoming NaN. 

This operation was performed so that no data was lost in the merging process. Even if values for certain years in Gender Wage Gap dataset were not available, the values from other indicators for that specific country-year combination, will still be assigned and saved.

In [ ]:
df_cleaned = pd.read_csv("data/outputs/wage_gap_transformed.csv")

# Full set of expected years:
expected_years = set(range(1970, 2025))

# Group by country and collect available years
years_by_country = df_cleaned.groupby("country")["year"].apply(lambda x: sorted(x.unique()))

# Find countries that do NOT have all years 1970–2024
countries_incomplete = {}

for country, years in years_by_country.items():
    years_set = set(years)
    missing_years = sorted(expected_years - years_set)
    if missing_years:
        countries_incomplete[country] = missing_years


# Creating a new dataset with rows for each country year combination 1970-2024

# Define all countries and all years 1970–2024
countries = sorted(df_cleaned["country"].unique())
years = list(range(1970, 2025))

# Create full country–year grid
full_index = pd.MultiIndex.from_product(
    [countries, years],
    names=["country", "year"]
)

# Reindex data to this full grid -> missing combos values become NaN
df_full_panel = (
    df_cleaned
    .set_index(["country", "year"])
    .reindex(full_index)
    .reset_index()
)

print("Shape of full panel (countries x years):", df_full_panel.shape)

df_full_panel.to_csv("data/outputs/wage_gap_full_grid.csv", index=False)


### 3.2 Data Merging

In this step, we build one final dataset by merging all thematic datasets onto the main target one (Gender Wage Gap). We started from it to ensure that merged dataset contain only observations for countries for which wage-gap data exists. 

Datasets were merged using left join. This keeps all wage-gap rows, while adding variables from other tables if the match was found. 

In [ ]:
# Load the main dataset: gender wage gap
wage = pd.read_csv("data/outputs/wage_gap_full_grid.csv")

# Make sure country and year are consistent
wage["country"] = wage["country"].str.strip()
wage["year"] = wage["year"].astype(int)

# Load other datasets
education = pd.read_csv("data/outputs/Education_gap.csv")
employment = pd.read_csv("data/outputs/Employment_gap.csv")
lfpr = pd.read_csv("data/outputs/lfp_gap.csv")
gdp = pd.read_csv("data/outputs/GDP.csv")
instit = pd.read_csv("data/outputs/GIS_gap.csv")
parttime = pd.read_csv("data/outputs/Part-time_employment_gap.csv")
public_emp = pd.read_csv("data/outputs/public_employment_representation_gap.csv")

# Standardise keys for each dataset
datasets = [education, employment, lfpr, gdp, instit, parttime, public_emp]

for i, df in enumerate(datasets):
    df["country"] = df["country"].str.strip()
    df["year"] = df["year"].astype(int)
    # Drop duplicates on (country, year) just in case
    datasets[i] = df.drop_duplicates(subset=["country", "year"])

# Start from wage gap and merge everything onto it
merged = wage.copy()

merged = merged.merge(education, on=["country", "year"], how="left")
merged = merged.merge(employment, on=["country", "year"], how="left")
merged = merged.merge(lfpr, on=["country", "year"], how="left")
merged = merged.merge(gdp, on=["country", "year"], how="left")
merged = merged.merge(instit, on=["country", "year"], how="left")
merged = merged.merge(parttime, on=["country", "year"], how="left")
merged = merged.merge(public_emp, on=["country", "year"], how="left")

merged.to_csv("data/outputs/merged_dataset.csv", index=False)


### 3.3 Reporting (Before and After Merging)

This block creates a summary table used for reporting. It calculates, for each dataset, the number of distinct countries and country–year observations, and also reports the total union coverage across all datasets as well as the final coverage after merging everything onto the wage-gap dataset.

In [ ]:
# Build a dict
data_dict = {
    "wage_gap": wage.copy(),
    "education_gap": education.copy(),
    "employment_gap": employment.copy(),
    "lfp_gap": lfpr.copy(),
    "gdp": gdp.copy(),
    "institutions_gap": instit.copy(),
    "part_time_gap": parttime.copy(),
    "public_employment_gap": public_emp.copy(),
}

# Standardise keys and drop duplicates for coverage calculations
for name, df in data_dict.items():
    df["country"] = df["country"].str.strip()
    df["year"] = df["year"].astype(int)
    data_dict[name] = df.drop_duplicates(subset=["country", "year"]).copy()

rows = []

# One row per dataset
for name, df in data_dict.items():
    n_countries = df["country"].nunique()
    n_country_year = df[["country", "year"]].drop_duplicates().shape[0]
    rows.append({
        "dataset": name,
        "n_distinct_countries": n_countries,
        "n_distinct_country_years": n_country_year,
    })

# Union across ALL datasets (before restricting to wage gap)
all_pairs = pd.concat(
    [df[["country", "year"]] for df in data_dict.values()],
    ignore_index=True
).drop_duplicates()

rows.append({
    "dataset": "Union across ALL datasets",
    "n_distinct_countries": all_pairs["country"].nunique(),
    "n_distinct_country_years": all_pairs.shape[0],
})

# Coverage after merging everything onto wage gap
merged_pairs = merged[["country", "year"]].drop_duplicates()

rows.append({
    "dataset": "Coverage after merging onto wage gap",
    "n_distinct_countries": merged_pairs["country"].nunique(),
    "n_distinct_country_years": merged_pairs.shape[0],
})

coverage_table = pd.DataFrame(rows)
print(coverage_table)

## 4. Data Profilling

To understand the structure of the newly created dataset, data profilling was performed and the report was generated with the following code.

In [ ]:
profile = ProfileReport(merged, title="YData Profiling Report")

profile.to_file("data/reports/report1.html")

## 5. Missing Data Imputation

### 5.1 Gender, Institutions and Development Indicators exclusion.

Due to the availability of data in Gender, Institutions and Development dataset only for 2023, and therefore very high number of missing values, the decision was made to drop indicators of this dataset and exclude them from our analysis for now.

In [ ]:
# Load datasets
merged = pd.read_csv("data/outputs/merged_dataset.csv")
gis = pd.read_csv("data/outputs/GIS_gap.csv")

# identify GIS columns except 'country' and 'year'
gis_columns = [col for col in gis.columns if col not in ["country", "year"]]

# print("GIS variables:", gis_columns)

# Create Dataset which excludes GIS variables

panel_no_gis = merged.drop(columns=gis_columns, errors="ignore")

print("Panel dataset shape (no GIS):", panel_no_gis.shape)

# Calculate percentage of missing values after dropping
total_cells = panel_no_gis.size
missing_cells = panel_no_gis.isna().sum().sum()
missing_percentage = (missing_cells / total_cells) * 100

print(f"Missing values (absolute): {missing_cells}")
print(f"Missing values (%): {missing_percentage:.2f}%")

# Save new dataset
panel_no_gis.to_csv("data/outputs/dataset_panel_no_GIS.csv", index=False)


### 5.2 Filtering for only specific years

#### 5.2.1 Plotting the number of non-missing observations over the years

To determine the appropriate start and end years for the analysis, the number of non-missing observations across time was computed and plotted.

This helped us to identify the range of years with sufficient data coverage, which for our dataset spans from 2010 to 2023.

In [ ]:
year_counts = (
    panel_no_gis
    .groupby("year")["wage_gap"]
    .apply(lambda s: s.notna().sum())
)

plt.figure(figsize=(12, 6))
plt.bar(year_counts.index, year_counts.values)
plt.title("Number of Available Wage Gap Observations per Year")
plt.xlabel("Year")
plt.ylabel("Number of non-missing wage gap values")
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.show()

#### 5.2.2 Dropping observations ouside of 2010-2023 range and removing aggregated entities.

After determining which years should be included in the analysis the remainig observations outside of the 2010-2023 range were dropped. 

Additionally, it was observed that some rows included aggregated values for European Union and OECD averages. These observations were also excluded from the analysis, as they do not represent distinct countries.  

In [ ]:
df = pd.read_csv("data/outputs/dataset_panel_no_GIS.csv")

# Filter to years 2010-2023
df_filt = df[(df["year"] >= 2010) & (df["year"] < 2024)].copy()

# Remove aggregated entities
countries_to_drop = [
    "European Union (19 countries) in OECD",
    "OECD",
    "European Union (27 countries)"
]

df_filt = df_filt[~df_filt["country"].isin(countries_to_drop)].copy()

# check results
print(df_filt["country"].unique())
print(df_filt.shape)

# Calculate percentage of missing values after dropping
total_cells = df_filt.size
missing_cells = df_filt.isna().sum().sum()
missing_percentage = (missing_cells / total_cells) * 100

print(f"Missing values (absolute): {missing_cells}")
print(f"Missing values (%): {missing_percentage:.2f}%")

# Save final dataset
df_filt.to_csv("data/outputs/dataset_filtered_2010.csv", index=False)

### 5.3 Removal of Indicators

#### 5.3.1 Plotting indicators against the number of missing values per indicator

To identify the indicators with the highest number of missing values, each indicator was plotted against its number of missing values.

In [ ]:
data = df_filt

# Exclude ID columns
id_cols = ["country", "year"]
indicator_cols = [col for col in data.columns if col not in id_cols]

# Count missing values per indicator
missing_counts = data[indicator_cols].isna().sum().sort_values()

# Plot horizontal bar chart
plt.figure(figsize=(12, 8))
plt.barh(missing_counts.index, missing_counts.values)

plt.title("Missing Values per Indicator (2010–2023, Filtered Dataset)", fontsize=15)
plt.xlabel("Number of Missing Values")
plt.grid(axis='x', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

#### 5.3.2 Dropping columns with more than 50% of missing values

"Gender gap in professional judges representation", "Gender gap in public sector employment", and "Gender representation in parliament", were identified as indicators with the highest number of missing values (more than 50%), and therefore the decision was made to exclude them from the analysis.

In [ ]:
df = pd.read_csv("data/outputs/dataset_filtered_2010.csv")

# Drop the three indicators with the highest missing values
cols_to_drop = [
    "Gender gap in professional judges representation",
    "Gender gap in public sector employment ",
    "Gender gap in representation in parliament"
]

df_reduced = df.drop(columns=cols_to_drop)

#### 5.3.3 Examining "Gender gap in senior management positions in national administrations" indicator

"Gender gap in senior management positions in national administrations" indicator required additioanl examination. 

To make a decision whether this variable should also be dropped, we have counted and dispalyed the countires for which there was no single data entry for this indicator across all years. After this we have manually checked, by looking at the data, if those countires correspond to those where a lof of data is missing also for the Gender Wage Gap variable. This was done to assess whether "Gender gap in senior management positions in national administrations" indicator or the countries with many missing values should be dropped. 

After a close examination, we came to the conclusion that "Gender gap in senior management positions in national administrations" indicator should be dropped. Most of the countries did not correspond to those mentioned earlier. Addiotnally, imputing values for missing data where there is no single value available would be both unreliable and methodologically inappropriate, as it would introduce artificial structure unsupported by the data.

In [ ]:
col = "Gender gap in senior management positions in national administrations"

# Count non-missing values per country for this indicator
counts = df_reduced.groupby("country")[col].apply(lambda s: s.notna().sum())

# Countries with zero observed values
countries_zero = counts[counts == 0].index.tolist()

print(f"Number of countries with no values for '{col}': {len(countries_zero)}")
print("\nCountries with zero values:")
for country in countries_zero:
    print(" -", country)

# Drop the indicator
df_reduced = df_reduced.drop(columns=col)

# Save final dataset
df_reduced.to_csv("data/outputs/dataset_cols_dropped.csv", index=False)

### 5.4 Removal of Countries

If some countries had too much data missing for multiple indicators, they had to be excluded from the analysis. The process of identifying those countries was both automatic (helped in faster identification) and manual (considering each case independently).

To address this dilemna the number of missing values for each country was calculated and plotted. Subsequently, for each indicator the countries with no observations for any year where displayed. This helped us to identify for which countries data was missing consistently across many indicators, and therefore which countries should be excluded from the analysis.

#### 5.4.1 Identifying for each country the number of missing values

In [ ]:
df = df_reduced

# Identify indicator columns
id_cols = ["country", "year"]
indicator_cols = [c for c in df.columns if c not in id_cols]

# Count total missing values per country
missing_counts = (
    df.groupby("country")[indicator_cols]
      .apply(lambda x: x.isna().sum().sum())
      .sort_values(ascending=False)
)

# Plot
plt.figure(figsize=(10, 12))
plt.barh(missing_counts.index, missing_counts.values)

plt.xlabel("Number of Missing Values")
plt.title("Missing Data per Country (Raw Counts)")
plt.grid(axis='x', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
# Dictionary to store results
missing_entirely = {}

# Loop through each indicator
for col in indicator_cols:
    # Count non-missing values for each country
    counts = df.groupby("country")[col].apply(lambda s: s.notna().sum())

    # Countries where all values are missing
    countries_no_data = counts[counts == 0].index.tolist()

    missing_entirely[col] = countries_no_data

# Display results
for col, countries in missing_entirely.items():
    print(f"\n{col} – Countries with no data at all: {len(countries)}")
    for c in countries:
        print("  -", c)

After a close examination we decided to remove Argentina, Brasil, Bulgaria. Croatia, Cyprus, India, Malta, Peru, Romania, and Turkey from the analysis. Most of these countries had data missing across many indicators. Turkey, had a lot of data missing in Gender Wage Gap indicator. Considering this is our target variable, it is exceptionally important it is accurate, therefore the decision to drop it.

In [ ]:
# List of countries to drop
countries_to_remove = [
    "Argentina",
    "Brazil",
    "Bulgaria",
    "Croatia",
    "Cyprus",
    "India",
    "Malta",
    "Peru",
    "Romania",
    "Türkiye"
]

# Drop
df_cleaned = df_reduced[~df_reduced["country"].isin(countries_to_remove)].copy()

# Save final dataset
df_cleaned.to_csv("data/outputs/dataset_countries_droped.csv", index=False)


### 5.3 Identification of incorrect values

While examining the dataset for some observations in Gender Wage Gap indicator values equal to exactly 0.0 where observed. These were identified to be extremely unlikely to be true values, and most likely resulted from errors during data collection or insertion, as values for these countries for other years were very far from zero. 

To confirm our suspicions we identified the exact observations with 0.0 values in gender wage gap and subsequently plotted, the distribution of this varible across years, for those countries.

#### 5.3.4 Gender Wage Gap distribution visualization

In [ ]:
# Check if any zeros exist
zero_count = (df_cleaned["wage_gap"] == 0).sum()
print(f"Number of EXACT zero values in wage_gap: {zero_count}")

# Show rows containing wage_gap = 0
if zero_count > 0:
    print("\nRows where wage_gap is exactly zero:")
    print(df_cleaned[df_cleaned["wage_gap"] == 0].head())
else:
    print("\nNo exact zeros found in wage_gap.")

# Visualizing the distribution of wage_gap

plt.figure(figsize=(14, 5))

# Histogram
plt.subplot(1, 2, 1)
sns.histplot(df_cleaned["wage_gap"], kde=True, bins=30)
plt.axvline(0, color='red', linestyle='--', label="Zero")
plt.title("Distribution of Wage Gap (Histogram + KDE)")
plt.xlabel("Wage Gap (%)")
plt.ylabel("Frequency")
plt.legend()

# Boxplot
plt.subplot(1, 2, 2)
sns.boxplot(x=df_cleaned["wage_gap"])
plt.axvline(0, color='red', linestyle='--', label="Zero")
plt.title("Boxplot of Wage Gap")
plt.xlabel("Wage Gap (%)")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Countries that appeared in the "wage_gap == 0" table
zero_countries = df_cleaned.loc[df_cleaned["wage_gap"] == 0, "country"].unique()

# Subset of the data for only those countries (all years, not just zeros)
df_subset = df_cleaned[df_cleaned["country"].isin(zero_countries)]

plt.figure(figsize=(10, 6))
sns.boxplot(data=df_subset, x="country", y="wage_gap")
plt.axhline(0, linestyle="--", color="red", label="Zero")
plt.title("Distribution of Wage Gap for Selected Countries")
plt.xlabel("Country")
plt.ylabel("Wage Gap (%)")
plt.legend()
plt.tight_layout()
plt.show()

#### 5.3.5 Insertion of NaN for identified observations

To deal with the identified problem the 0.0 values were replaced with Nans.

In [ ]:
df_cleaned["wage_gap"] = df_cleaned["wage_gap"].replace(0.0, np.nan)

# Save final dataset
df_cleaned.to_csv("data/outputs/dataset_0_handled.csv", index=False)

### 5.4 Interpolation

To impute some of the missing values linear interpolation was applied. It was choosen as an appropiate technique because many missing data was not there by chance. As it was displayed on a graph in section (5.2), many countries reported some of the indicators only every 2 or 4 years. As a result, the underlying trajectories of the variables are typically smooth and gradual rather than abrupt. Linear interpolation therefore provides a reasonable approximation of the unobserved values between reporting years.

The values were only interpolated within country, only after first non-NaN value.


In [ ]:
# Load
df_before = pd.read_csv("data/outputs/dataset_0_handled.csv")

# Sort
df_before = df_before.sort_values(["country", "year"]).reset_index(drop=True)

# Identify numeric columns except 'year'
numeric_cols = df_before.select_dtypes(include="number").columns.tolist()
if "year" in numeric_cols:
    numeric_cols.remove("year")

def interpolate_country(group: pd.DataFrame) -> pd.DataFrame:
    group = group.sort_values("year")

    # interpolate all numeric columns
    for col in numeric_cols:
        group[col] = group[col].interpolate(
            method="linear",
            limit_direction="forward"
        )

    return group

# Apply interpolation
df_after = (
    df_before
    .groupby("country", group_keys=False)
    .apply(interpolate_country)
    .reset_index(drop=True)
)

df_after.to_csv("data/outputs/dataset_interpolated.csv", index=False)

#### 5.4.1 Visualising how much data was interpolated per country

In [ ]:

# Align both dataframes on country-year index
before_num = (
    df_before
    .set_index(["country", "year"])[numeric_cols]
)

after_num = (
    df_after
    .set_index(["country", "year"])[numeric_cols]
)


interp_mask = before_num.isna() & after_num.notna()

# Count interpolated cells per country (sum over years and variables)
interp_counts = (
    interp_mask
    .groupby(level="country")
    .sum()         # sum per column
    .sum(axis=1)   # sum across all numeric columns
    .sort_values(ascending=False)
)

# Keep only countries where at least one value was interpolated
interp_counts_nonzero = interp_counts[interp_counts > 0]

print("Number of interpolated cells per country:")
print(interp_counts_nonzero)

# Plot total number of interpolated values per country
plt.figure(figsize=(10, 8))
plt.barh(interp_counts_nonzero.index, interp_counts_nonzero.values)
plt.xlabel("Number of interpolated values")
plt.title("Amount of Interpolated Data per Country")
plt.grid(axis="x", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

### 5.5 Bfill

To impute values that occurred before the first year with reported data, a backward fill (bfill) method was applied. This approach replaces missing values at the beginning of each country’s time series with the first available non-missing observation. 

This method was not applied for United States, Japan, Korea, Latvia, Lithuania, Costa Rica and Colombia, as these countries had too many years of missing data before the first value available, or one indicator was entirely unavailable, making this imputation method inappropriate for those countries.

In [ ]:
# Countries to exclude from backward fill
countries_no_bfill = [
    "United States",
    "Japan",
    "Korea",
    "Latvia",
    "Lithuania",
    "Costa Rica",
    "Colombia"
]

df_bfill = df_after.copy()

# Identify numeric columns
numeric_cols = df_bfill.select_dtypes(include="number").columns.tolist()
if "year" in numeric_cols:
    numeric_cols.remove("year")

# Function applied per country
def bfill_country(group: pd.DataFrame) -> pd.DataFrame:
    country = group["country"].iloc[0]

    # Skip backward fill for specified countries
    if country in countries_no_bfill:
        return group

    # Apply bfill to all numeric columns
    group[numeric_cols] = group[numeric_cols].bfill()

    return group

# Apply per country
df_final = (
    df_bfill
    .groupby("country", group_keys=False)
    .apply(bfill_country)
    .reset_index(drop=True)
)

df_final.to_csv("data/outputs/dataset_bfill.csv", index=False)

#### 5.5.1 Visualization of how much data was inserted with bfilled

In [ ]:
# Make sure both are aligned and use only numeric columns
before_bfill = (
    df_after
    .set_index(["country", "year"])[numeric_cols]
)

after_bfill = (
    df_final
    .set_index(["country", "year"])[numeric_cols]
)


bfill_mask = before_bfill.isna() & after_bfill.notna()

# Count bfilled cells per country (sum over years and variables)
bfill_counts = (
    bfill_mask
    .groupby(level="country")
    .sum()         # sum per column
    .sum(axis=1)   # sum across all numeric columns
    .sort_values(ascending=False)
)

# Keep only countries where at least one value was bfilled
bfill_counts_nonzero = bfill_counts[bfill_counts > 0]

print("Number of values filled by bfill per country:")
print(bfill_counts_nonzero)

# Plot total number of bfilled values per country
plt.figure(figsize=(10, 8))
plt.barh(bfill_counts_nonzero.index, bfill_counts_nonzero.values)
plt.xlabel("Number of values filled by bfill")
plt.title("Amount of Backward-Filled Data per Country")
plt.grid(axis="x", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

### 5.6 Data Imputation for United States, Japan, Korea, Latvia, Lithuania, Costa Rica and Colombia

#### 5.6.1 Data Imputation for United States, Japan, Korea, Latvia, Lithuania

Japan, Korea, Latvia, and Lithuania were missing entirely or almost entirely (across all years) values in labour force participation indicators. For United States on the other hand there was no data for part-time gap indicator. 

As these countries had usually no missing values for other indicators, providing a lot of interesting insides to our analysis, we decided not to exclude them. To deal with the mentioned problem, we decided to fill the missing values with the averages of countries with similar economic structure, institutional context, and labour-market characteristics, and for which little or no data was imputed earlier.

For Latvia and Lithuania these were: Poland, Czechia, Estonia, and Finland.

For Japan and Korea: Canada, Australia, New Zealand, United States and United Kingdom. - Other high-income, industrialized OECD non-european countries.

For United States: Canada, Australia, New Zealand, Japan, Korea and United Kingdom.

Even if some values for later years were available, these observations were overwritten with the group averages as well. This ensured that the imputed series is fully consistent across all years for each affected country and no mixing of sporadic real observations with synthetic values constructed from a different data-generating process, was done. 

By assigning to these countries the average of a carefully chosen comparison group, the future model will treat them as having labour-market characteristics similar to those of other well-developed economies, diminishing the potential bias. 


In [ ]:
df_before = df_final.sort_values(["country", "year"]).reset_index(drop=True)

# Work on a copy
df_imputed = df_before.copy()

# Define groups and variables to impute
gap_cols = ["native_gap", "immigrant_gap", "overall_gap"]

# Groups for Japan & Korea
group_JK = ["United States", "Canada", "Australia", "New Zealand", "United Kingdom"]

# Groups for Lithuania & Latvia
group_LL = ["Poland", "Czechia", "Estonia", "Finland"]

# Group for US part_time_gap
group_US_pt = ["Japan", "Korea", "Canada", "Australia", "New Zealand", "United Kingdom"]

# Compute group means for native_gap & immigrant_gap & overall_gap
means_JK = df_imputed[df_imputed["country"].isin(group_JK)][gap_cols].mean()
means_LL = df_imputed[df_imputed["country"].isin(group_LL)][gap_cols].mean()

print("Means used for Japan & Korea (from US, CA, AU, NZ, UK):")
print(means_JK)
print("\nMeans used for Lithuania & Latvia (from PL, CZ, EE, FI):")
print(means_LL)

# Impute native_gap & immigrant_gap for Japan & Korea
countries_JK = ["Japan", "Korea"]
df_imputed.loc[df_imputed["country"].isin(countries_JK), gap_cols] = means_JK.values

# Impute for Lithuania & Latvia
countries_LL = ["Lithuania", "Latvia"]
df_imputed.loc[df_imputed["country"].isin(countries_LL), gap_cols] = means_LL.values

# Impute part_time_gap for United States
pt_mean_US = df_imputed[df_imputed["country"].isin(group_US_pt)]["part_time_gap"].mean()
print("\nMean used for US part_time_gap (from JP, KR, CA, AU, NZ, UK):", pt_mean_US)

df_imputed.loc[df_imputed["country"] == "United States", "part_time_gap"] = pt_mean_US


#### 5.6.2 Costa Rica and Colombia observations dropping

Although the initiall stategy for data imputation for Costa Rica and Colombia was the same as for the United States, Japan, Korea, Latvia, Lithuania, unfortunatelly no similar or close countries to Colombia and Costa Rica where identified in the dataset. The only other South-American countries included in the dataset were Mexico and Chile, with Chile already having a lot of imputed values itself. Considering this problem we decided to exclude Costa Rica and Colombia from the analysis, as including them could potentially cause too much noise in the data. 

In [ ]:
countries_to_drop = ["Costa Rica", "Colombia"]

df_clean = df_imputed[~df_imputed["country"].isin(countries_to_drop)].copy()

## 6. Correlation Analysis

In [ ]:
# Compute correlation matrix
corr_matrix = df_clean.corr(numeric_only=True)

# Plot heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=False, cmap="coolwarm", center=0)
plt.title("Correlation Matrix of Indicators", fontsize=16)
plt.tight_layout()
plt.show()


### 6.1 Examining correlation between immigrant_gap and native_gap

#### 6.1.1 Plotting Immigrant Gap vs Native Gap

In [ ]:
native_col = "native_gap"
immigrant_col = "immigrant_gap"

# Create scatterplot
plt.figure(figsize=(8,6))
plt.scatter(df_clean[native_col], df_clean[immigrant_col], alpha=0.6)

plt.xlabel("Native Gap")
plt.ylabel("Immigrant Gap")
plt.title("Scatterplot of Immigrant Gap vs. Native Gap")
plt.grid(True, linestyle="--", alpha=0.4)

plt.show()

#### 6.1.2 Overall Gap indicator dropping

Immigrant_gap and native_gap are two indicators representing gender gap in labour force participation across immigrants and natives respectively. Overall_gap indicator, on the other hand, represents combined, gap in labour force participation across the two groups.

After plotting the two variables we discovered that although correlated, the correlation is not too high, and both immigrant_gap and native_gap indicators should be included in the analysis, as they can provide interesting insights to it. Consequently, Overall Gap indicator should be dropped, as it does not add any new information to the analysis.

In [ ]:
df_clean = df_clean.drop(columns=["overall_gap"])

df_clean.to_csv("data/outputs/FINAL_DATASET.csv", index=False)


## 7. Cleaned Data Profilling

In [ ]:
profile = ProfileReport(df_clean, title="YData Profiling Report")

profile.to_file("data/reports/report2.html")

## 8. Modelling

In [ ]:
# Reload preprocessed data set

df = pd.read_csv("data/outputs/FINAL_DATASET.csv")

display(df.dtypes)
display(df.describe())

In [ ]:
target = "wage_gap"

features = [
    "education_gap",
    "employment_gap",
    "native_gap",
    "immigrant_gap",
    "gdp_pc",
    "part_time_gap",
]

### 8.1 Creation of the train and test datasets

In [ ]:
# We chose a 70-30 split because we want to ensure that the test set is not too small
# so that the results of the model evaluation are meaningful

train_size = 0.7

In [ ]:
# Split data to be used for model training and for testing

X = df[features].values
y = df[target].values

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=train_size, shuffle=True)

print(
    f"Row count in training set: {X_train.shape[0]}",
    f"Row count in test set:     {X_test.shape[0]}",
    f"Proportion:                {round(X_train.shape[0] / (X_train.shape[0] + X_test.shape[0]), 2)}",
    sep="\n",
)

In [ ]:
# Prepare a scaled version of the input features for use by the neural network model
# Since some features present a right-skewed distribution, we use MinMaxScaler

scaler = MinMaxScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(
    f"Input feature minimum values: scaled from {np.min(X_train)} to {np.min(X_train_scaled)}",
    f"Input feature maximum values: scaled from {np.max(X_train)} to {np.max(X_train_scaled)}",
    sep="\n",
)

### 8.2 Train a Decision Tree model

Decision Trees are predictive models that split data into branches based on feature values. At each node, the algorithm chooses the feature and split point that best separate the data, typically by minimizing variance (in the case of regression). The process continues recursively until stopping criteria are met. Predictions are made from the values in the final leaf nodes.

#### 8.2.1 Create baseline model

In [ ]:
model = DecisionTreeRegressor()
model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
rmse_train = mean_squared_error(y_train, y_pred_train) ** 0.5

y_pred_test = model.predict(X_test)
rmse_test = mean_squared_error(y_test, y_pred_test) ** 0.5

print(
    f"RMSE on train dataset: {rmse_train:.4f}",
    f"RMSE on test dataset: {rmse_test:.4f}",
    sep="\n",
)

#### 8.2.2 Tune hyperparameters

The model trained with the default hyperparameters is overfitting the data, as evidenced by the big difference between the RMSE in the train and test sets.

We will now do a cross-validated random search to determine reasonable model hyperparameters.

The initial search space was determined based on "common sense":
* max_depth: Determines how deep we will allow the tree to grow. Setting 9 as the maximum to keep overfitting in check.
* min_samples_leaf: Minimum number of observations required per final leaf node. Setting a minimum of 5 to prevent overfitting.
* We will leave other hyperparameters with default values.


In [ ]:
param_search_space = {
    "max_depth": list(range(2, 10)),
    "min_samples_leaf": list(range(5, 20)),
}

model = DecisionTreeRegressor()

random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_search_space,
    n_iter=100,
    scoring="neg_root_mean_squared_error",
    cv=3,
)

random_search.fit(X_train, y_train)

# RandomizedSearchCV returns the negative of the RMSE because it needs to maximize it
best_score = abs(random_search.best_score_)
best_params = random_search.best_params_

print(
    f"Best hyperparameters: {best_params}",
    f"Best RMSE score:      {best_score}",
    sep="\n",
)

In [ ]:
# Train and evaluate model with the best hyperparameters found

model = DecisionTreeRegressor(**best_params)
model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
rmse_train = mean_squared_error(y_train, y_pred_train) ** 0.5

y_pred_test = model.predict(X_test)
rmse_test = mean_squared_error(y_test, y_pred_test) ** 0.5

print(
    f"RMSE on train dataset: {rmse_train:.4f}",
    f"RMSE on test dataset: {rmse_test:.4f}",
    sep="\n",
)

These error metrics are more reasonable, although the gap between training set and test set scores is still suspicious, as is the fact that the random search selected the largest max_depth we permitted it. We will revisit this after reviewing the model features.

#### 8.2.3 Feature selection

In [ ]:
# Display feature importances

importances = model.feature_importances_
indices = np.argsort(importances)
feature_names = [features[i] for i in indices]

plt.title("Feature importances for the Decision Tree")
plt.barh(feature_names, importances[indices])
plt.ylabel("Features")
plt.xlabel("Importance")
plt.show()

For the sake of experimentation towards model simplification, we will remove the "native_gap" feature and retrain the model.

We detected that this feature showed some correlation with "employment_gap" during the exploration phase. Since the model is assigning a similar importance score to both, one of them could be redundant.

In [ ]:
# Train and evaluate model with the best hyperparameters found, removing "native_gap"" from input features

last_col = features.index("native_gap")

new_X_train = np.delete(X_train, last_col, axis=1)
new_X_test = np.delete(X_test, last_col, axis=1)

model = DecisionTreeRegressor(**best_params)
model.fit(new_X_train, y_train)

y_pred_train = model.predict(new_X_train)
rmse_train = mean_squared_error(y_train, y_pred_train) ** 0.5

y_pred_test = model.predict(new_X_test)
rmse_test = mean_squared_error(y_test, y_pred_test) ** 0.5

print(
    f"RMSE on train dataset: {rmse_train:.4f}",
    f"RMSE on test dataset: {rmse_test:.4f}",
    sep="\n",
)

In [ ]:
# Display new feature importances

importances = model.feature_importances_
indices = np.argsort(importances)
feature_names = [[f for f in features if f != "native_gap"][i] for i in indices]

plt.title("Feature importances for the Decision Tree")
plt.barh(feature_names, importances[indices])
plt.ylabel("Features")
plt.xlabel("Importance")
plt.show()

Model performance improved slightly after removing that feature.

We will now remove "immigrant_gap" since it also showed some correlation with "employment_gap".

In [ ]:
# Train and evaluate model with the best hyperparameters found, removing "native_gap" from input features

cols_to_remove = (features.index("native_gap"), features.index("immigrant_gap"))
new_X_train = np.delete(X_train, cols_to_remove, axis=1)
new_X_test = np.delete(X_test, cols_to_remove, axis=1)

model = DecisionTreeRegressor(**best_params)
model.fit(new_X_train, y_train)

y_pred_train = model.predict(new_X_train)
rmse_train = mean_squared_error(y_train, y_pred_train) ** 0.5

y_pred_test = model.predict(new_X_test)
rmse_test = mean_squared_error(y_test, y_pred_test) ** 0.5

print(
    f"RMSE on train dataset: {rmse_train:.4f}",
    f"RMSE on test dataset: {rmse_test:.4f}",
    sep="\n",
)

In [ ]:
# Display new feature importances

importances = model.feature_importances_
indices = np.argsort(importances)
feature_names = [[f for f in features if f not in ("native_gap", "immigrant_gap")][i] for i in indices]

plt.title("Feature importances for the Decision Tree")
plt.barh(feature_names, importances[indices])
plt.ylabel("Features")
plt.xlabel("Importance")
plt.show()

Model performance is still comparable to the previously obtained best score. Since removing this feature results in a simpler model, we will choose this option.

#### 8.2.4 Reexamine model hyperparameters

We will now double check the value of that "suspicious" max_depth parameter by tracking model performance in train and test sets for a range of possible values for this hyperparameter.


In [ ]:
errors_per_tree_depth = {}
model_candidates = {}

for d in [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]:
    candidate = DecisionTreeRegressor(min_samples_leaf=5, max_depth=d)
    candidate.fit(new_X_train, y_train)

    y_pred_train = candidate.predict(new_X_train)
    y_pred_test = candidate.predict(new_X_test)

    model_candidates[d] = candidate
    errors_per_tree_depth[d] = (
        mean_squared_error(y_train, y_pred_train) ** 0.5,
        mean_squared_error(y_test, y_pred_test) ** 0.5,
    )

plot_x = sorted(list(errors_per_tree_depth.keys()))
plot_y_train = [errors_per_tree_depth[x][0] for x in plot_x]
plot_y_test = [errors_per_tree_depth[x][1] for x in plot_x]

plt.plot(plot_x, plot_y_train, label="Training data")
plt.plot(plot_x, plot_y_test, label="Test data")
plt.legend()
plt.title("Evaluation metrics per tree max_depth")
plt.xlabel("Value of max_depth")
plt.ylabel("RMSE")
plt.show()

Performance on the test set stabilizes around max_depth = 7 so we will use this as the value for our model.

In [ ]:
# Train and evaluate model with max_depth = 7 - all input features

model = DecisionTreeRegressor(min_samples_leaf=5, max_depth=7)
model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
rmse_train = mean_squared_error(y_train, y_pred_train) ** 0.5

y_pred_test = model.predict(X_test)
rmse_test = mean_squared_error(y_test, y_pred_test) ** 0.5

print(
    f"RMSE on train dataset: {rmse_train:.4f}",
    f"RMSE on test dataset: {rmse_test:.4f}",
    sep="\n",
)

In [ ]:
# Display new feature importances

importances = model.feature_importances_
indices = np.argsort(importances)
feature_names = [features[i] for i in indices]

plt.title("Feature importances for the Decision Tree")
plt.barh(feature_names, importances[indices])
plt.ylabel("Features")
plt.xlabel("Importance")
plt.show()

In [ ]:
# Train and evaluate model with max_depth = 7 - using top input features only

model = DecisionTreeRegressor(min_samples_leaf=5, max_depth=7)
model.fit(new_X_train, y_train)

y_pred_train = model.predict(new_X_train)
rmse_train = mean_squared_error(y_train, y_pred_train) ** 0.5

y_pred_test = model.predict(new_X_test)
rmse_test = mean_squared_error(y_test, y_pred_test) ** 0.5

print(
    f"RMSE on train dataset: {rmse_train:.4f}",
    f"RMSE on test dataset: {rmse_test:.4f}",
    sep="\n",
)

In [ ]:
# Display new feature importances

importances = model.feature_importances_
indices = np.argsort(importances)
feature_names = [[f for f in features if f not in ("native_gap", "immigrant_gap")][i] for i in indices]

plt.title("Feature importances for the Decision Tree")
plt.barh(feature_names, importances[indices])
plt.ylabel("Features")
plt.xlabel("Importance")
plt.show()

Performance is comparable or better than observed in previous results.

In [ ]:
# Display the finalist Decision Tree

plt.figure(figsize=(40, 10))
plot_tree(
    model,
    feature_names=feature_names,
    filled=True,
    rounded=True,
    fontsize=10,
)
plt.title("Decision Tree Structure")
plt.show()

#### 8.2.5 Save final model

In [ ]:
best_tree = model.copy()

with open("models/best_decision_tree.pkl", "wb") as fp:
    pickle.dump(best_tree, fp)

## 8.3 Train a Neural Network model

In [ ]:
# Convert datasets to 2D PyTorch tensors

X_train_nn = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_nn = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
X_test_nn = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_nn = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

# Inputs excluding "immigrant_gap" and "native_gap" features

cols_to_remove = (features.index("native_gap"), features.index("immigrant_gap"))
new_X_train_scaled = np.delete(X_train_scaled, cols_to_remove, axis=1)
new_X_test_scaled = np.delete(X_test_scaled, cols_to_remove, axis=1)

new_X_train_nn = torch.tensor(new_X_train_scaled, dtype=torch.float32)
new_X_test_nn = torch.tensor(new_X_test_scaled, dtype=torch.float32)

#### 8.3.1 Create baseline model

In [ ]:
lr = 0.005  # model learning rate
n_hidden_layers = 1  # model architecture
n_epochs = 200  # number of training iterations
batch_size = 20  # training data batches

model, history = train_neural_network(
    X_train=X_train_nn,
    y_train=y_train_nn,
    X_test=X_test_nn,
    y_test=y_test_nn,
    lr=lr,
    n_hidden_layers=n_hidden_layers,
    n_epochs=n_epochs,
    batch_size=batch_size,
)

#### 8.3.2 Feature selection

In [ ]:
lr = 0.005  # model learning rate
n_hidden_layers = 1  # model architecture
n_epochs = 200  # number of training iterations
batch_size = 20  # training data batches

model, history = train_neural_network(
    X_train=new_X_train_nn,
    y_train=y_train_nn,
    X_test=new_X_test_nn,
    y_test=y_test_nn,
    lr=lr,
    n_hidden_layers=n_hidden_layers,
    n_epochs=n_epochs,
    batch_size=batch_size,
)

The model evaluation metrics worsened slightly, but not by a large enough gp that it cannot be attributed to the random nature of the model.

It is somewhat expected that a neural networks would perform better with more input data.

We will continue evaluating the effect that removing these two features has on the data in the next steps.

#### 8.3.3 Model tuning and experimenting with architecture

In [ ]:
lr_candidates = (0.01, 0.007, 0.003, 0.001, 0.0001)
n_hidden_layers_candidates = (1, 2, 3)
n_epochs = 600
batch_size = 20
without_immigrant_or_native_gap = (True, False)

best_nn_model = None
best_model_history = None
best_params = {}

all_results = []

for n_hidden_layers in n_hidden_layers_candidates:
    for lr in lr_candidates:
        for reduced_feature_set in without_immigrant_or_native_gap:

            print(f"Training NN with {n_hidden_layers} hidden layers / learning rate = {lr} / reduced feature set = {reduced_feature_set}...")

            if reduced_feature_set:
                X_train_tensor = new_X_train_nn
                X_test_tensor = new_X_test_nn
            else:
                X_train_tensor = X_train_nn
                X_test_tensor = X_test_nn

            model, history = train_neural_network(
                X_train=X_train_tensor,
                y_train=y_train_nn,
                X_test=X_test_tensor,
                y_test=y_test_nn,
                lr=lr,
                n_hidden_layers=n_hidden_layers,
                n_epochs=n_epochs,
                batch_size=batch_size,
                print_results=False,
            )

            if best_model_history is None or min(best_model_history) > min(history):
                best_nn_model = model
                best_model_history = history
                best_params["lr"] = lr
                best_params["n_hidden_layers"] = n_hidden_layers
                best_params["reduced_feature_set"] = reduced_feature_set

            all_results.append(
                {
                    "lr": lr,
                    "n_hidden_layers": n_hidden_layers,
                    "reduced_feature_set": reduced_feature_set,
                    "rmse": min(history),
                }
            )

print(
    "\nBest model configuration found:",
    f"    Hidden layers: {best_params['n_hidden_layers']}",
    f"    Learning rate: {best_params['lr']}",
    f"    Reduced feature set: {best_params['reduced_feature_set']}",
    f"    Error metrics (RMSE): {min(best_model_history)}",
    sep="\n",
)

In [ ]:
plt.plot(best_model_history)
plt.title("Model metrics evolution")
plt.xlabel("Epoch")
plt.ylabel("RMSE")
plt.show()

In [ ]:
search_learning_rate = np.array([float(res["lr"]) for res in all_results])
search_hidden_layers = np.array([int(res["n_hidden_layers"]) for res in all_results])
search_feature_set = np.array([int(res["reduced_feature_set"]) for res in all_results])
search_rmse = np.array([float(res["rmse"]) for res in all_results])

visualize_3_param_grid_search_results(
    search_hidden_layers, "# Hidden layers",
    search_feature_set, "Uses reduced feature set",
    search_learning_rate, "Learning rate",
    search_rmse
)

The best performing model uses all 6 input features, an architecture of 3 hidden layers and a high-end learning rate (0.007).

In [ ]:
# Save model

torch.save(best_nn_model.state_dict(), "models/best_nn_weights.pkl")

## 9. Result inspection using LIME

### 9.1 Decision Tree

In [ ]:
feature_names = [f for f in features if f not in ("native_gap", "immigrant_gap")]

explainer = LimeTabularExplainer(
    training_data=new_X_train,
    feature_names=feature_names,
    mode="regression",
)

In [ ]:
# Visualize some test data points

for i in [0, 10, 20]:

    inputs = new_X_test[i].reshape(1, -1)
    true_target = y_test[i]
    pred_target = best_tree.predict(inputs)[0]

    print(
        f"True wage gap:   {true_target}",
        f"Predicted value: {pred_target}",
        sep="\n",
    )

    exp = explainer.explain_instance(
        inputs.flatten(),
        best_tree.predict,
        num_features=len(feature_names),
    )

    print([(f, float(x)) for f, x in zip(feature_names, inputs.flatten())])
    print(exp.as_list())

    fig = exp.as_pyplot_figure(label=true_target)
    plt.title("LIME explanation")
    plt.tight_layout()
    plt.show()

### 9.2 Neural network

In [ ]:
explainer = LimeTabularExplainer(
    training_data=X_train_scaled,
    feature_names=features,
    mode="regression",
)

In [ ]:
# Visualize some test data points

best_nn_model.eval()

for i in [0, 10, 20]:

    inputs = X_test_scaled[i]
    nn_inputs = X_test_nn[i]
    true_target = y_test_nn[i][0]
    pred_target = best_nn_model(nn_inputs)[0]

    print(
        f"True wage gap:   {true_target}",
        f"Predicted value: {pred_target}",
        sep="\n",
    )

    def wrapped_predict(data):
        return best_nn_model(torch.tensor(data, dtype=torch.float32)).detach().numpy()

    exp = explainer.explain_instance(
        inputs,
        wrapped_predict,
        num_features=len(feature_names),
    )

    print([(f, float(x)) for f, x in zip(features, inputs)])
    print(exp.as_list())

    fig = exp.as_pyplot_figure(label=true_target)
    plt.title("LIME explanation")
    plt.tight_layout()
    plt.show()